# EgoLife Two-User Video-First QA Generation (Colab)

This notebook runs the updated EgoLife two-user QA pipeline from `Long-video-understanding`.

Current flow:

1. Build an EgoLife video/gaze manifest for two users.
2. Prepare aligned evidence packets with videos, sampled frames, and gaze summaries.
3. Generate multi-user MCQ questions with the video-first loop.
4. Judge and evaluate whether the question really needs two users.
5. Validate and export the generated questions.

The newest pipeline is `generate_video_qa_loop`. It is different from the older `observe_clips -> mine_candidates -> generate_qa` debug path.

## 1. Colab Setup

Run this first. It clones the repo branch and installs the packages needed for Hugging Face downloads, video/frame handling, and optional Qwen3-VL inference.

In [ ]:
# Colab setup. Re-run safely if the runtime restarts.
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/KangsanKim07/Long-video-understanding.git'
BRANCH = 'main'  # latest video-first pipeline is on main; change if needed
REPO_DIR = Path('/content/Long-video-understanding') if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        %cd {REPO_DIR}
        !git fetch origin {BRANCH}
        !git checkout {BRANCH}
        !git pull --ff-only origin {BRANCH} || true
    %cd {REPO_DIR}
else:
    print('Not in Colab; using current directory:', REPO_DIR)

# Core dependencies. qwen-vl-utils/decord are used by local Qwen3-VL video inference.
!pip install -q datasets huggingface_hub pandas pillow tqdm imageio opencv-python-headless qwen-vl-utils decord accelerate "transformers>=4.57.0"

sys.path.insert(0, str(REPO_DIR))
print('Repo:', REPO_DIR)

## 2. Configure the Run

For a quick Colab test, keep `TARGET_COUNT = 10` and two users. Increase later after the first successful run.

`DRY_RUN = True` only builds prompts and checks the pipeline wiring; it does not call a model. Set it to `False` in the model cells below when you are ready to generate questions.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import textwrap
import pandas as pd

OUTPUT_ROOT = Path('egolife_two_user_qa/outputs/colab_video_first')
CACHE_DIR = OUTPUT_ROOT / 'hf_cache'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DAYS = 'DAY1'
AGENTS = 'A1_JAKE,A2_ALICE'  # two EgoLife users
MAX_PER_AGENT_DAY = 8        # number of clips scanned per user/day while building manifest
TARGET_COUNT = 10            # desired number of accepted questions
USERS_PER_CASE = 2
FRAMES_PER_CLIP = 5          # sampled frames saved per clip for fallback/API modes
MAX_ATTEMPTS = 3

MANIFEST = OUTPUT_ROOT / 'manifest.json'
EVIDENCE = OUTPUT_ROOT / 'evidence_manifest.jsonl'
QA_OUTPUT = OUTPUT_ROOT / 'qa_mcq.video_first.jsonl'
PROMPTS_OUTPUT = OUTPUT_ROOT / 'video_first_prompts.jsonl'
REJECTED_OUTPUT = OUTPUT_ROOT / 'video_first_rejected.jsonl'
REPORT_OUTPUT = OUTPUT_ROOT / 'generation_report.md'
CSV_OUTPUT = OUTPUT_ROOT / 'qa_mcq.csv'

print('Output root:', OUTPUT_ROOT.resolve())
print('Agents:', AGENTS)
print('Target accepted questions:', TARGET_COUNT)

## 3. Helper Functions

In [ ]:
def run_cmd(cmd):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)

def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def show_questions(path=QA_OUTPUT):
    rows = read_jsonl(path)
    if not rows:
        print('No rows yet:', path)
        return pd.DataFrame()
    flat = []
    for r in rows:
        flat.append({
            'qa_id': r.get('qa_id'),
            'question_type': r.get('question_type'),
            'category': r.get('category'),
            'question': r.get('question'),
            'correct': r.get('correct'),
            'answer': r.get('answer'),
            'required_users': ', '.join(r.get('required_users', [])),
            'attempt_count': r.get('attempt_count'),
            'judge_status': (r.get('review') or {}).get('status'),
            'why_two_users_needed': r.get('why_two_users_needed'),
        })
    df = pd.DataFrame(flat)
    display(df)
    return df

## 4. Build Manifest

This queries the EgoLife Hugging Face dataset metadata and builds a local manifest for the selected users/day.

In [ ]:
run_cmd([
    sys.executable, '-m', 'egolife_two_user_qa', 'build_manifest',
    '--days', DAYS,
    '--agents', AGENTS,
    '--max-per-agent-day', MAX_PER_AGENT_DAY,
    '--output', MANIFEST,
])

with MANIFEST.open('r', encoding='utf-8') as f:
    manifest = json.load(f)
print('Manifest clips:', len(manifest.get('clips', [])))
print(json.dumps(manifest.get('clips', [])[:2], indent=2)[:2000])

## 5. Prepare Evidence Packets

This aligns two users by day/time token and caches videos plus sampled frames. The sampled frames are important because API-compatible backends often accept images more reliably than raw video files.

In [ ]:
run_cmd([
    sys.executable, '-m', 'egolife_two_user_qa', 'prepare_evidence',
    '--manifest', MANIFEST,
    '--output', EVIDENCE,
    '--cache-dir', CACHE_DIR,
    '--output-root', OUTPUT_ROOT,
    '--target-count', TARGET_COUNT,
    '--users-per-case', USERS_PER_CASE,
    '--frames-per-clip', FRAMES_PER_CLIP,
])

evidence_rows = read_jsonl(EVIDENCE)
print('Evidence packets:', len(evidence_rows))
if evidence_rows:
    preview = evidence_rows[0].copy()
    print(json.dumps(preview, indent=2)[:4000])

## 6. Dry Run: Generate Prompts Only

Use this before model inference. It verifies that the evidence packets can be turned into video-first generation, judge, and answerability prompts.

In [ ]:
DRY_PROMPTS = OUTPUT_ROOT / 'video_first_prompts.dryrun.jsonl'
DRY_OUTPUT = OUTPUT_ROOT / 'qa_mcq.video_first.dryrun.jsonl'
DRY_REJECTED = OUTPUT_ROOT / 'video_first_rejected.dryrun.jsonl'

run_cmd([
    sys.executable, '-m', 'egolife_two_user_qa', 'generate_video_qa_loop',
    '--evidence', EVIDENCE,
    '--output', DRY_OUTPUT,
    '--prompts-output', DRY_PROMPTS,
    '--rejected-output', DRY_REJECTED,
    '--target-count', min(2, TARGET_COUNT),
    '--max-attempts', 1,
    '--dry-run',
])

prompt_rows = read_jsonl(DRY_PROMPTS)
print('Prompt rows written:', len(prompt_rows))
if prompt_rows:
    print(json.dumps(prompt_rows[0], indent=2)[:4000])

## 7A. Real Generation with an OpenAI-Compatible API

Use this when you have an API endpoint that supports OpenAI-style `/chat/completions` with image input. The code uses sampled frames by default. Only enable `ALLOW_OPENAI_VIDEO_INPUT=True` if your endpoint truly supports video payloads.

For OpenRouter, set `BASE_URL = "https://openrouter.ai/api/v1"` and put your key in `LOCAL_VLM_API_KEY` when prompted.

In [ ]:
from getpass import getpass

# Example choices. Change MODEL_ID to any multimodal model your endpoint supports.
BASE_URL = 'https://openrouter.ai/api/v1'
MODEL_ID = 'google/gemini-2.5-flash'
ALLOW_OPENAI_VIDEO_INPUT = False  # False = use sampled frames; True = send local videos if server supports video URLs
RUN_API_GENERATION = False        # set True after entering an API key

if RUN_API_GENERATION:
    if not os.environ.get('LOCAL_VLM_API_KEY'):
        os.environ['LOCAL_VLM_API_KEY'] = getpass('LOCAL_VLM_API_KEY: ')

    cmd = [
        sys.executable, '-m', 'egolife_two_user_qa', 'generate_video_qa_loop',
        '--backend', 'openai-compatible-local',
        '--base-url', BASE_URL,
        '--model-id', MODEL_ID,
        '--evidence', EVIDENCE,
        '--output', QA_OUTPUT,
        '--prompts-output', PROMPTS_OUTPUT,
        '--rejected-output', REJECTED_OUTPUT,
        '--target-count', TARGET_COUNT,
        '--max-attempts', MAX_ATTEMPTS,
        '--max-new-tokens', 1536,
    ]
    if ALLOW_OPENAI_VIDEO_INPUT:
        cmd.append('--allow-openai-video-input')
    run_cmd(cmd)
    show_questions(QA_OUTPUT)
else:
    print('Set RUN_API_GENERATION = True to call the API backend.')

## 7B. Optional: Real Generation with Local Qwen3-VL on Colab GPU

This path loads `Qwen/Qwen3-VL-8B-Instruct` directly with Transformers. Use a GPU runtime. A T4 may run out of memory; L4/A100 is safer.

In [ ]:
RUN_LOCAL_QWEN3VL = False
LOCAL_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'

if RUN_LOCAL_QWEN3VL:
    run_cmd([
        sys.executable, '-m', 'egolife_two_user_qa', 'generate_video_qa_loop',
        '--backend', 'transformers-local',
        '--model-id', LOCAL_MODEL_ID,
        '--dtype', 'bfloat16',
        '--evidence', EVIDENCE,
        '--output', QA_OUTPUT,
        '--prompts-output', PROMPTS_OUTPUT,
        '--rejected-output', REJECTED_OUTPUT,
        '--target-count', TARGET_COUNT,
        '--max-attempts', MAX_ATTEMPTS,
        '--max-new-tokens', 1536,
    ])
    show_questions(QA_OUTPUT)
else:
    print('Set RUN_LOCAL_QWEN3VL = True to run local Qwen3-VL inference on a GPU runtime.')

## 8. Inspect Accepted and Rejected Questions

In [ ]:
df = show_questions(QA_OUTPUT)

rejected_rows = read_jsonl(REJECTED_OUTPUT)
print('Rejected attempts:', len(rejected_rows))
if rejected_rows:
    sample = rejected_rows[0]
    print(json.dumps(sample, indent=2)[:3000])

## 9. Validate and Export

Run this after real generation. The validator checks schema and the two-user answerability gate.

In [ ]:
if QA_OUTPUT.exists() and read_jsonl(QA_OUTPUT):
    run_cmd([
        sys.executable, '-m', 'egolife_two_user_qa', 'validate_outputs',
        '--qa', QA_OUTPUT,
        '--csv-output', CSV_OUTPUT,
        '--report', REPORT_OUTPUT,
        '--strict-review',
    ])
    print('CSV:', CSV_OUTPUT.resolve())
    print('Report:', REPORT_OUTPUT.resolve())
    print(REPORT_OUTPUT.read_text(encoding='utf-8')[:4000])
else:
    print('No accepted QA output yet. Run one of the real generation cells first.')

## 10. Download Outputs from Colab

In [ ]:
if IN_COLAB:
    from google.colab import files
    for path in [QA_OUTPUT, PROMPTS_OUTPUT, REJECTED_OUTPUT, CSV_OUTPUT, REPORT_OUTPUT]:
        path = Path(path)
        if path.exists():
            print('Downloading', path)
            files.download(str(path))
else:
    print('Not in Colab. Output files are under:', OUTPUT_ROOT.resolve())

## Notes

- `question_type = commonality` means the generated question asks about something that requires comparing/combining both users' experiences.
- `question_type = difference` means the generated question asks about a contrast between the users' experiences.
- The answerability gate rejects questions if a single user's evidence is enough or if the combined users cannot answer it.
- Without Aria calibration, gaze remains summarized as yaw/pitch/depth statistics rather than projected 2D image coordinates.